In [ ]:
dataset = 'https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Toys_and_Games_5.json.gz'

In [75]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
def create_spark_session():
    """Create a Spark Session"""
    _ = load_dotenv()
    return (
        SparkSession
        .builder
        .appName("SparkApp")
        .master("local[5]")
        .config("spark.driver.memory", "16g")
        .config("spark.executor.memory", "16g")
        .getOrCreate()
    )
# spark = create_spark_session()
print('Session Started')
print('Code Executed Successfully')

Session Started
Code Executed Successfully


In [3]:
import pandas as pd
from tqdm import tqdm
import json
PATH_BIGDATA = '/opt/spark-notebooks/pandas_to_pyspark/Toys_and_Games_5.json'
def read_json_to_pdf(path: str) -> pd.DataFrame: 
  data = []
  with open(path, 'r') as f: 
    for line in tqdm(f):
      data.append(json.loads(line))
  df = pd.DataFrame(data)
  return df
raw_pdf = read_json_to_pdf(PATH_BIGDATA)
print(raw_pdf.head())

print('Code Executed Successfully')

1828971it [00:36, 50191.77it/s]


   overall vote  verified   reviewTime      reviewerID        asin  \
0      5.0    3      True   10 6, 2013  A2LSCFZM2FBZK7  0486427706   
1      5.0    9      True   08 9, 2013  A3IXP5VS847GE5  0486427706   
2      5.0  NaN      True   04 5, 2016  A1274GG1EB2JLJ  0486427706   
3      5.0    3      True  02 13, 2016  A30X5EGBYAZQQK  0486427706   
4      5.0  NaN      True  12 10, 2015  A3U6UNXLAUY6ZV  0486427706   

                       style                     reviewerName  \
0  {'Format:': ' Paperback'}                           Ginger   
1  {'Format:': ' Paperback'}  Dragonflies &amp; Autumn Leaves   
2  {'Format:': ' Paperback'}                      barbara ann   
3  {'Format:': ' Paperback'}                         Samantha   
4  {'Format:': ' Paperback'}                      CP in Texas   

                                          reviewText  \
0  The stained glass pages are pretty cool. And i...   
1  My 11 y.o. loved this...and so do I (you know ...   
2  The pictures are 

In [4]:

raw_pdf = pd.read_json(PATH_BIGDATA, orient='records',lines=True)
print(raw_pdf.head())
print('Code Executed Successfully')

   overall vote  verified   reviewTime      reviewerID        asin  \
0        5    3      True   10 6, 2013  A2LSCFZM2FBZK7  0486427706   
1        5    9      True   08 9, 2013  A3IXP5VS847GE5  0486427706   
2        5  NaN      True   04 5, 2016  A1274GG1EB2JLJ  0486427706   
3        5    3      True  02 13, 2016  A30X5EGBYAZQQK  0486427706   
4        5  NaN      True  12 10, 2015  A3U6UNXLAUY6ZV  0486427706   

                       style                     reviewerName  \
0  {'Format:': ' Paperback'}                           Ginger   
1  {'Format:': ' Paperback'}  Dragonflies &amp; Autumn Leaves   
2  {'Format:': ' Paperback'}                      barbara ann   
3  {'Format:': ' Paperback'}                         Samantha   
4  {'Format:': ' Paperback'}                      CP in Texas   

                                          reviewText  \
0  The stained glass pages are pretty cool. And i...   
1  My 11 y.o. loved this...and so do I (you know ...   
2  The pictures are 

In [5]:
# implement tqdm
all_data=[]
for i in tqdm(open(PATH_BIGDATA,'r')):
    all_data.append(i)

1828971it [00:22, 82277.68it/s] 


In [6]:
import pyspark

In [7]:
spark=create_spark_session()


In [8]:
spark.conf.set("spark.sql.caseSensitive", "true") 
raw_sdf=spark.read.json(PATH_BIGDATA)

26/01/05 13:37:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
COL_NAME_MAP = {
    "overall": "overall",
    "verified": "verified",
    "reviewTime": "review_time",
    "reviewerID": "reviewer_id",
    "asin": "asin",
    "reviewerName": "reviewer_name",
    "reviewText": "review_text",
    "summary": "summary",
    "unixReviewTime": "unix_review_time",
    "style": "style",
    "vote": "vote",
    "image": "image"
}
print('Initial Columns names:')
i=1
for col_name in raw_pdf.columns: 
    print(f'{i}: {col_name}')
    i=i+1

## renaming column names
raw_pdf = raw_pdf.rename(columns=COL_NAME_MAP)

print('___________________________')
print('Columns names after rename:')
i=1
for col_name in raw_pdf.columns: 
    print(f'{i}: {col_name}')
    i=i+1

print('___________________________')
print('Code Executed Successfully')



Initial Columns names:
1: overall
2: vote
3: verified
4: reviewTime
5: reviewerID
6: asin
7: style
8: reviewerName
9: reviewText
10: summary
11: unixReviewTime
12: image
___________________________
Columns names after rename:
1: overall
2: vote
3: verified
4: review_time
5: reviewer_id
6: asin
7: style
8: reviewer_name
9: review_text
10: summary
11: unix_review_time
12: image
___________________________
Code Executed Successfully


In [10]:
raw_sdf.schema.names

['asin',
 'image',
 'overall',
 'reviewText',
 'reviewTime',
 'reviewerID',
 'reviewerName',
 'style',
 'summary',
 'unixReviewTime',
 'verified',
 'vote']

In [11]:
COL_NAME_MAP = {
    "overall": "overall",
    "verified": "verified",
    "reviewTime": "review_time",
    "reviewerID": "reviewer_id",
    "asin": "asin",
    "reviewerName": "reviewer_name",
    "reviewText": "review_text",
    "summary": "summary",
    "unixReviewTime": "unix_review_time",
    "style": "style",
    "vote": "vote",
    "image": "image"
}

print('___________________________')
print('Columns names after rename:')
i=1
for col_name in raw_sdf.schema.names: 
    print(f'{i}: {col_name}')
    i=i+1

def rename_columns(df, column_map):
  for old, new in column_map.items():
        df = df.withColumnRenamed(old, new)
  return df

  
raw_sdf = rename_columns(raw_sdf, COL_NAME_MAP)
print('___________________________')
print('Columns names after rename:')
i=1
for col_name in raw_sdf.schema.names: 
    print(f'{i}: {col_name}')
    i=i+1

print('___________________________')

print('Code Executed Successfully')


___________________________
Columns names after rename:
1: asin
2: image
3: overall
4: reviewText
5: reviewTime
6: reviewerID
7: reviewerName
8: style
9: summary
10: unixReviewTime
11: verified
12: vote
___________________________
Columns names after rename:
1: asin
2: image
3: overall
4: review_text
5: review_time
6: reviewer_id
7: reviewer_name
8: style
9: summary
10: unix_review_time
11: verified
12: vote
___________________________
Code Executed Successfully


In [12]:
SELECTED_COLUMNS = [
    "reviewer_id",
    "asin",
    "review_text",
    "summary",
    "verified",
    "overall",
    "vote",
    "unix_review_time",
    "review_time",
]

# Alternative method:
# raw_sdf = raw_sdf.select("reviewer_id", "asin", ...) 
raw_sdf = raw_sdf.select(*SELECTED_COLUMNS)
print(raw_sdf.show())

print('Code Executed Successfully')

+--------------+----------+--------------------+--------------------+--------+-------+----+----------------+-----------+
|   reviewer_id|      asin|         review_text|             summary|verified|overall|vote|unix_review_time|review_time|
+--------------+----------+--------------------+--------------------+--------+-------+----+----------------+-----------+
|A2LSCFZM2FBZK7|0486427706|The stained glass...|           Nice book|    true|    5.0|   3|      1381017600| 10 6, 2013|
|A3IXP5VS847GE5|0486427706|My 11 y.o. loved ...|      Great pictures|    true|    5.0|   9|      1376006400| 08 9, 2013|
|A1274GG1EB2JLJ|0486427706|The pictures are ...|The pictures are ...|    true|    5.0|NULL|      1459814400| 04 5, 2016|
|A30X5EGBYAZQQK|0486427706|I absolutely love...|       So beautiful!|    true|    5.0|   3|      1455321600|02 13, 2016|
|A3U6UNXLAUY6ZV|0486427706|          I love it!|          Five Stars|    true|    5.0|NULL|      1449705600|12 10, 2015|
|A1SAJF5SNM6WJS|0486427706|MY HU

In [13]:
# import time

# def create_path_snapshot():
#     path_fixed = '/data/snapshot/pandas/data_{}.json' 
#     current_unix_time = int(time.time())
#     return path_fixed.format(current_unix_time)

# PATH_SNAPSHOT = create_path_snapshot()
# raw_pdf.to_json(PATH_SNAPSHOT)
# print('Snapshot Saved')
# print('Code Executed Successfully')

In [14]:
# import time

# def create_path_snapshot():
#     path_fixed = 'data/snapshot/pyspark/snapshot_{}' 
#     current_unix_time = int(time.time())
#     return path_fixed.format(current_unix_time)

# PATH_SNAPSHOT = create_path_snapshot()
# raw_sdf = (
# raw_sdf
#     .repartition("asin")
#     .sortWithinPartitions("unix_review_time")
# )
# # Write data with partition and sorted
# (
# raw_sdf
#     .write.partitionBy("asin")
#     .mode("overwrite")
#     .parquet(PATH_SNAPSHOT)
# )

# print('Code Executed Successfully')

In [15]:
# #cache
# raw_sdf = spark.read.json(PATH_BIGDATA).cache()
# raw_sdf.count()
raw_sdf.show()

+--------------+----------+--------------------+--------------------+--------+-------+----+----------------+-----------+
|   reviewer_id|      asin|         review_text|             summary|verified|overall|vote|unix_review_time|review_time|
+--------------+----------+--------------------+--------------------+--------+-------+----+----------------+-----------+
|A2LSCFZM2FBZK7|0486427706|The stained glass...|           Nice book|    true|    5.0|   3|      1381017600| 10 6, 2013|
|A3IXP5VS847GE5|0486427706|My 11 y.o. loved ...|      Great pictures|    true|    5.0|   9|      1376006400| 08 9, 2013|
|A1274GG1EB2JLJ|0486427706|The pictures are ...|The pictures are ...|    true|    5.0|NULL|      1459814400| 04 5, 2016|
|A30X5EGBYAZQQK|0486427706|I absolutely love...|       So beautiful!|    true|    5.0|   3|      1455321600|02 13, 2016|
|A3U6UNXLAUY6ZV|0486427706|          I love it!|          Five Stars|    true|    5.0|NULL|      1449705600|12 10, 2015|
|A1SAJF5SNM6WJS|0486427706|MY HU

In [16]:
spark.catalog.clearCache()


In [19]:
import yaml
from pathlib import Path
from pyspark.sql import SparkSession, DataFrame


def load_latest_parquet_path() -> Path:
    """
    :return: Path to the latest parquet file storage
    """
    with open("versions.yaml", "r") as f:
        content = yaml.safe_load(f)
        latest = content["latest"]
        path = content[latest]["path"]
        return Path(path)


def read_latest_snapshot(ctx: SparkSession) -> DataFrame:
    """
    Read parquet data source from latest metadata

    :param ctx: SparkSession
    :return: PySpark DataFrame
    """
    path = load_latest_parquet_path()
    df = ctx.read.parquet(str(path))
    return df


main_df = read_latest_snapshot(spark)
main_df.show()


FileNotFoundError: [Errno 2] No such file or directory: 'versions.yaml'